# LPGNet — exact replication (Exp 1) & leakage-free protocol (Exp 2)

**Reference:** Alle & Priyakumar (2021), *"LPGNet: A lightweight architecture for
the diagnosis of Parkinson's disease using gait signals"* (arXiv:2107.12878).

**Dataset:** PhysioNet `gaitpdb` 1.0.0 — gait in Parkinson's disease
(306 recordings, 93 PD + 73 control subjects, vertical ground-reaction force,
18 channels, 100 Hz, ~2 min walks).

This notebook runs two experiments under **identical, fixed 10-fold stratified
subject-level splits**:

| | LP fit / normalisation | leakage? |
|---|---|---|
| **Exp 1** — replication | **global**: fit once on *all* control recordings (paper protocol, confirmed in their `generateLPresidual.m`) | yes — test-fold controls shape their own features |
| **Exp 2** — corrected | **fold-local**: refit inside each fold on *train-fold* controls only | no |

The leak is **one-sided**: only *control* recordings influence the LP predictors
and normalisation, so any inflation should appear mainly in **specificity**.

**Reported paper values to compare against (Exp 1):**
AUC **91.7 ± 9.4**, ACC **90.3 ± 5.8**, F1 **93.2 ± 3.6** (percent).

*Runtime: ~20 small trainings; minutes on a Colab GPU, longer on CPU.*


In [ ]:
# ============================================================
# Setup & global knobs
# ============================================================
import os, time, random, json, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import lfilter
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- knobs (edit freely) ----
N_FOLDS        = 10      # outer subject-level CV folds (paper: 10)
LP_ORDER       = 11      # linear-predictor order (paper: 11)
WIN, STRIDE    = 100, 50 # 2 s windows @50 Hz, 50% overlap (paper)
BATCH_WIN      = 128     # window-stage batch size (paper: 128)
LR_WIN         = 5e-4    # window-stage learning rate (paper: 5e-4)
LABEL_SMOOTH   = 0.1     # BCE label smoothing (paper: 0.1)
L2_REG         = 1e-4    # paper does NOT report the L2 value -> 1e-4 chosen (see notes)
MAX_EPOCHS_WIN = 60      # hard cap; early stopping usually fires much earlier
PATIENCE       = 8
VAL_FRAC       = 0.15    # fraction of TRAIN SUBJECTS held out for early stopping

print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================================
# Download & parse PhysioNet gaitpdb
# (306 recordings / 166 subjects: 93 PD, 73 control; 18 VGRF channels, 100 Hz)
# ============================================================
URL = "https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip"
ZIP = "gaitpdb.zip"; DATA_DIR = "gaitpdb_data"
if not os.path.exists(DATA_DIR):
    if not os.path.exists(ZIP):
        print("downloading ~300 MB (one-off) ...")
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)
ROOT = os.path.join(DATA_DIR, "gait-in-parkinsons-disease-1.0.0")

records = []
for fn in sorted(os.listdir(ROOT)):
    if not fn.endswith(".txt") or fn[2:4] not in ("Co", "Pt"):
        continue
    base = fn[:-4]
    arr = np.loadtxt(os.path.join(ROOT, fn), delimiter="\t")
    sig = arr[:, 1:19].astype(np.float32)      # drop time column -> 18 VGRF channels
    records.append(dict(file=fn,
                        study=base[:2],                     # Ga / Ju / Si
                        subject=base.split("_")[0],         # e.g. GaPt15
                        label=0 if base[2:4] == "Co" else 1,  # 0=control, 1=PD
                        signal=sig))

subjects   = sorted({r["subject"] for r in records})
subj_label = {s: next(r["label"] for r in records if r["subject"] == s) for s in subjects}
n_pd  = sum(1 for s in subjects if subj_label[s] == 1)
n_co  = len(subjects) - n_pd
print(f"{len(records)} recordings | {len(subjects)} subjects ({n_pd} PD, {n_co} control)")
lens = np.array([len(r["signal"]) for r in records])
print(f"recording length @100 Hz: min {lens.min()}, median {int(np.median(lens))}, max {lens.max()}")

In [ ]:
# ============================================================
# Preprocessing (paper): causal 2-tap moving average, then decimate 100 -> 50 Hz
# Linear Prediction residuals: order 11, ONE predictor per channel, fit on
# concatenated CONTROL recordings with `order` zeros padding in between
# (this mirrors the paper's generateLPresidual.m exactly).
# ============================================================
def preprocess(x):
    return lfilter([0.5, 0.5], 1, x, axis=0)[::2]

def fit_lp(control_sigs, order=LP_ORDER):
    """Fit one order-`order` LP predictor per channel on concatenated controls.
    Returns (coeffs [18, order], stds [18]) -- stds = per-channel std of the
    concatenated control SIGNAL, used for normalisation, as in the paper.
    Autocorrelation via FFT (same values as the paper's xcorr, O(n log n))."""
    C, conc_all = np.zeros((18, order)), []
    for c in range(18):
        conc = np.concatenate([np.concatenate([s[:, c],
                                               np.zeros(order, np.float32)])
                               for s in control_sigs])
        conc_all.append(conc)
        n = len(conc)
        nfft = 1 << (2 * n - 1).bit_length()
        f = np.fft.rfft(conc, nfft)
        r = np.fft.irfft(f * np.conj(f), nfft)[:order + 1] / n   # biased autocorr
        R = np.array([[r[abs(i-j)] for j in range(order)] for i in range(order)])
        C[c] = np.linalg.solve(R, r[1:order+1])
    stds = np.std(np.stack(conc_all, axis=1), axis=0) + 1e-8
    return C, stds

def apply_lp(x, C, stds):
    """Prediction-error filter: e(n) = x(n) - sum_i c_i x(n-i), normalised."""
    e = np.stack([lfilter(np.r_[1.0, -C[c]], 1, x[:, c]) for c in range(18)], axis=1)
    return (e / stds).astype(np.float32)

def set_feats(idx, C, stds, key="feat"):
    """Compute LP-residual features for a set of recordings in-place."""
    for i in idx:
        records[i][key] = apply_lp(records[i]["proc"], C, stds)

for r in records:                       # preprocessed signals, computed once
    r["proc"] = preprocess(r["signal"])

# --- sanity check: raw VGRF vs LP residual for one control & one PD recording ---
demo_C, demo_stds = fit_lp([r["proc"] for r in records if r["label"] == 0])
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, lab, ttl in [(axes[0], 0, "control"), (axes[1], 1, "PD")]:
    r = next(r for r in records if r["label"] == lab)
    e = apply_lp(r["proc"], demo_C, demo_stds)
    ax.plot(r["proc"][:500, 8] / r["proc"][:, 8].std(), label="raw VGRF ch9 (z)")
    ax.plot(e[:500, 8], label="LP residual ch9")
    ax.set_title(f"{ttl} ({r['file']})"); ax.legend(); ax.set_xlabel("samples @50 Hz")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Fixed 10-fold STRATIFIED SUBJECT-LEVEL splits.
# Fixed seed -> every experiment in this notebook uses IDENTICAL folds,
# so configurations are directly comparable fold-by-fold.
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
subj_arr = np.array(subjects)
subj_y   = np.array([subj_label[s] for s in subjects])
FOLDS = []
for tr_s, te_s in skf.split(subj_arr, subj_y):
    tr_subj, te_subj = set(subj_arr[tr_s]), set(subj_arr[te_s])
    FOLDS.append(dict(
        train=[i for i, r in enumerate(records) if r["subject"] in tr_subj],
        test =[i for i, r in enumerate(records) if r["subject"] in te_subj]))
for k, f in enumerate(FOLDS):
    ntr = len({records[i]["subject"] for i in f["train"]})
    nte = len({records[i]["subject"] for i in f["test"]})
    print(f"fold {k}: {len(f['train'])} train recs ({ntr} subj) | "
          f"{len(f['test'])} test recs ({nte} subj)")

In [ ]:
# ============================================================
# LPGNet classifier (Fig. 2 of the paper)
#   SpatialDropout(0.2)
#   -> SepConv 7x32 -> BN -> ELU -> MaxPool
#   -> SepConv 5x32 -> BN -> ELU -> MaxPool
#   -> SepConv 3x32 -> BN -> ELU -> GlobalAveragePooling
#   -> Dropout(0.25) -> Dense(1, sigmoid)
# (MaxPool size is not reported; pool size 2 assumed.)
# ============================================================
def build_lpgnet(T, n_ch=18, channels=32, kernels=(7, 5, 3),
                 spat_drop=0.2, drop=0.25, l2=L2_REG):
    reg = keras.regularizers.l2(l2)
    # NOTE: variable-length input (None) -> the same network can process
    # 100-sample windows (step 1) AND whole recordings (step 2).
    inp = layers.Input((None, n_ch))
    x = layers.SpatialDropout1D(spat_drop)(inp)
    for i, k in enumerate(kernels):
        x = layers.SeparableConv1D(channels, k, padding="same",
                                   depthwise_regularizer=reg,
                                   pointwise_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("elu")(x)
        if i < len(kernels) - 1:
            x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D(name="pooled")(x)
    x = layers.Dropout(drop)(x)
    out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)(x)
    return keras.Model(inp, out)

_m = build_lpgnet(WIN)
print(f"LPGNet trainable params (window input): {_m.count_params():,}")
print("Paper reports 4,735 CNN params (4,933 incl. the 198 LP coefficients).")
print("Minor differences vs this Keras rebuild can come from unspecified")
print("details (pool size, padding, BN parameters, where L2 is applied).")
del _m

In [ ]:
# ============================================================
# Two-step training pipeline (paper, Sec. II-C)
#  step 1: train the whole network on 2 s windows (50% overlap)
#  step 2: freeze the conv backbone, recalibrate the FC layer on WHOLE
#          recordings. We implement step 2 exactly: run the frozen backbone
#          over each whole recording -> 32-d GAP embedding -> logistic
#          regression (mathematically the same objective as retraining
#          Dense(1,sigmoid) with cross-entropy, but deterministic).
# ============================================================
def make_windows(idx, key="feat"):
    Xw, yw = [], []
    for i in idx:
        s = records[i][key]
        for st in range(0, len(s) - WIN + 1, STRIDE):
            Xw.append(s[st:st + WIN]); yw.append(records[i]["label"])
    return np.asarray(Xw, np.float32), np.asarray(yw, np.float32)

def split_val(idx, frac=VAL_FRAC, seed=SEED):
    """Hold out a fraction of TRAIN SUBJECTS for early stopping
    (subject-level, stratified when both classes are present)."""
    ss = sorted({records[i]["subject"] for i in idx})
    yy = [subj_label[s] for s in ss]
    if len(set(yy)) > 1:
        sss = StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
        tr_s, va_s = next(sss.split(np.zeros(len(ss)), yy))
    else:
        rng = np.random.default_rng(seed)
        perm = rng.permutation(len(ss))
        n_va = max(1, int(round(frac * len(ss))))
        va_s, tr_s = perm[:n_va], perm[n_va:]
    tr_subj = {ss[i] for i in tr_s}; va_subj = {ss[i] for i in va_s}
    return ([i for i in idx if records[i]["subject"] in tr_subj],
            [i for i in idx if records[i]["subject"] in va_subj])

def train_window_stage(build_fn, train_idx, seed=SEED, lr=LR_WIN, verbose=0,
                       key="feat"):
    tf.keras.utils.set_random_seed(seed)
    tr_i, va_i = split_val(train_idx)
    Xtr, ytr = make_windows(tr_i, key=key); Xva, yva = make_windows(va_i, key=key)
    model = build_fn(WIN)
    model.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping("val_loss", patience=PATIENCE,
                                        restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.25, patience=3)]
    model.fit(Xtr, ytr, validation_data=(Xva, yva), batch_size=BATCH_WIN,
              epochs=MAX_EPOCHS_WIN, callbacks=cb, verbose=verbose)
    return model

def embedding_model(model):
    """Frozen backbone up to the pooled representation."""
    return keras.Model(model.input, model.get_layer("pooled").output)

def recalibrate_fc(model, train_idx, key="feat"):
    """Step 2: frozen backbone -> embedding -> logistic regression."""
    emb = embedding_model(model)
    E = np.stack([emb.predict(records[i][key][None], verbose=0)[0] for i in train_idx])
    y = np.array([records[i]["label"] for i in train_idx])
    clf = LogisticRegression(max_iter=2000)
    clf.fit(E, y)
    return clf

def predict_recordings(model, clf, idx, key="feat"):
    """Whole-recording inference -> PD probability per recording."""
    emb = embedding_model(model)
    return np.asarray([clf.predict_proba(emb.predict(records[i][key][None],
                       verbose=0)[0][None])[0, 1] for i in idx])

def predict_window_avg(model, idx, key="feat"):
    """Alternative inference: average window probabilities over a recording."""
    out = []
    for i in idx:
        s = records[i][key]
        ws = np.stack([s[st:st + WIN] for st in range(0, len(s) - WIN + 1, STRIDE)])
        out.append(float(model.predict(ws, verbose=0).mean()))
    return np.asarray(out)

def compute_metrics(y_true, p):
    y_hat = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return dict(AUC=roc_auc_score(y_true, p),
                ACC=accuracy_score(y_true, y_hat),
                F1=f1_score(y_true, y_hat),          # PD = positive class (paper)
                SENS=tp / (tp + fn), SPEC=tn / (tn + fp))

## Experiment 1 — replication attempt (paper protocol, **global** LP fit)

LP predictors and the normalisation stds are fitted **once on all control
recordings** — exactly what the paper's released code does — and then used for
every fold. Everything else follows the paper: order-11 LP, moving-average +
decimation to 50 Hz, 2 s windows with 50% overlap, batch 128, lr 5e-4, BCE with
label smoothing 0.1, two-step training (windows → whole-recording FC
recalibration), 10-fold stratified subject-level CV.

> **Why this leaks:** the LP filter of channel *c* is estimated from the
> autocorrelation of *all* control recordings, including subjects sitting in the
> test fold. Test controls are therefore whitened/normalised with statistics
> they contributed to. PD recordings never enter the fit, so the inflation is
> expected on the control side (specificity).

In [ ]:
# ============================================================
# EXPERIMENT 1 - replication attempt (GLOBAL LP fit = paper protocol)
# ============================================================
exp1_rows, t0 = [], time.time()
global_C, global_stds = fit_lp([r["proc"] for r in records if r["label"] == 0])
set_feats(range(len(records)), global_C, global_stds)   # global features for all

for k, f in enumerate(FOLDS):
    tk = time.time()
    model = train_window_stage(build_lpgnet, f["train"], seed=SEED + k)
    clf   = recalibrate_fc(model, f["train"])
    p     = predict_recordings(model, clf, f["test"])
    m     = compute_metrics([records[i]["label"] for i in f["test"]], p)
    exp1_rows.append(dict(fold=k, **m))
    print(f"fold {k}: AUC {m['AUC']:.3f}  ACC {m['ACC']:.3f}  F1 {m['F1']:.3f}  "
          f"SENS {m['SENS']:.3f}  SPEC {m['SPEC']:.3f}   ({time.time()-tk:.0f}s)")

exp1 = pd.DataFrame(exp1_rows)
print(f"\ntotal time: {(time.time()-t0)/60:.1f} min")
paper = dict(AUC=(91.7, 9.4), ACC=(90.3, 5.8), F1=(93.2, 3.6))
cmp1 = pd.DataFrame(
    {"replicated mean±std": [f"{exp1[m].mean()*100:.1f}±{exp1[m].std()*100:.1f}"
                              for m in ("AUC", "ACC", "F1")],
     "paper reported":      [f"{v[0]}±{v[1]}" for v in paper.values()]},
    index=["AUC", "ACC", "F1"])
display(exp1.round(3)); display(cmp1)

## Experiment 2 — leakage-free protocol (**fold-local** LP fit)

Identical folds, model, and training — the **only** change is that the LP
coefficients and normalisation stds are refit **inside each fold**, using
train-fold control recordings only. Test recordings are transformed with
statistics that never saw them.

In [ ]:
# ============================================================
# EXPERIMENT 2 - leakage-free protocol (FOLD-LOCAL LP fit)
# ============================================================
exp2_rows, t0 = [], time.time()
for k, f in enumerate(FOLDS):
    tk = time.time()
    C, stds = fit_lp([records[i]["proc"] for i in f["train"]
                      if records[i]["label"] == 0])       # train-fold controls only
    set_feats(f["train"] + f["test"], C, stds)
    model = train_window_stage(build_lpgnet, f["train"], seed=SEED + k)
    clf   = recalibrate_fc(model, f["train"])
    p     = predict_recordings(model, clf, f["test"])
    m     = compute_metrics([records[i]["label"] for i in f["test"]], p)
    exp2_rows.append(dict(fold=k, **m))
    print(f"fold {k}: AUC {m['AUC']:.3f}  ACC {m['ACC']:.3f}  F1 {m['F1']:.3f}  "
          f"SENS {m['SENS']:.3f}  SPEC {m['SPEC']:.3f}   ({time.time()-tk:.0f}s)")

exp2 = pd.DataFrame(exp2_rows)
print(f"\ntotal time: {(time.time()-t0)/60:.1f} min")
display(exp2.round(3))
print("fold-local LP mean±std:",
      {m: f"{exp2[m].mean():.3f}±{exp2[m].std():.3f}"
       for m in ("AUC", "ACC", "F1", "SENS", "SPEC")})

## Comparison — how much did the leak inflate the results?

In [ ]:
# ============================================================
# Exp 1 (leaked) vs Exp 2 (corrected) - paired comparison on the SAME folds
# ============================================================
metrics = ("AUC", "ACC", "F1", "SENS", "SPEC")
cmp = pd.DataFrame(
    {"Exp1 global LP (leaked)": [f"{exp1[m].mean():.3f}±{exp1[m].std():.3f}" for m in metrics],
     "Exp2 fold-local LP":      [f"{exp2[m].mean():.3f}±{exp2[m].std():.3f}" for m in metrics],
     "Wilcoxon p": [wilcoxon(exp1[m], exp2[m]).pvalue if (exp1[m] != exp2[m]).any()
                    else np.nan for m in metrics]},
    index=["AUC", "ACC", "F1", "Sensitivity", "Specificity"])
display(cmp)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, m in zip(axes, metrics):
    dfp = pd.DataFrame({"global LP (leaked)": exp1[m], "fold-local LP": exp2[m]})
    sns.boxplot(data=dfp, ax=ax, width=0.5)
    sns.stripplot(data=dfp, ax=ax, color="k", size=4, alpha=0.6)
    ax.set_title(m); ax.set_ylim(0.4, 1.02)
plt.tight_layout(); plt.show()

dlt = (exp1.set_index("fold")[list(metrics)] - exp2.set_index("fold")[list(metrics)])
print("per-fold delta (Exp1 - Exp2), mean:",
      {m: f"{dlt[m].mean():+.3f}" for m in metrics})
print("If the leak matters, deltas should be positive, driven by SPECIFICITY.")

## Notes & caveats

- **Subject-level split** is correctly implemented in both experiments: folds are
  built on subjects, and all recordings of a subject stay together.
- **L2 strength** is not reported in the paper; we use 1e-4 (flagged in the
  knobs cell). MaxPool size is not reported either; pool size 2 is assumed.
- **Step-2 recalibration** as implemented (frozen backbone → logistic
  regression on the GAP embedding) has the same objective as retraining the
  final Dense(1, sigmoid) layer with cross-entropy, but is deterministic.
- **Run-to-run variance** is non-negligible on a dataset this small
  (166 subjects); single-run differences of a few points between Exp 1 and
  Exp 2 are within noise. The paired Wilcoxon test across the 10 matched folds
  is the meaningful comparison.
- If the leak matters, **Exp 1 > Exp 2 especially on specificity**, because
  only control recordings enter the LP fit.
- Parameter count printed above may differ slightly from the reported 4,735
  (unspecified implementation details); the architecture follows Fig. 2 of the
  paper layer by layer.
